# LCA Lasers4MaaS: joining processes

Unit processes of the joining step for 5 use cases, with State of the Art (SoA) technologies and with the Lasers4MaaS solution (L4M-solution).
The functional unit is 1 joined unit (`1 unit`).

The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [1]:
import bw2data as bd
from IPython.display import display
from l4m_lca import plots
from l4m_lca.constants import (L4M_SOLUTION_DATABASE, L4M_SOLUTION_TECHNOLOGY, EU27_GB, EUROSTAT_ELEC_PRICE_2025S1, JOINING_STEM,
                               REGIONAL_BASELINE, REGIONAL_ELEC_PRICE, SOA_DATABASE)
from l4m_lca.costs import COST_COLUMNS, COST_UNITS, regional_labour_costs
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.joining import joining_processes, technology_of
from l4m_lca.lcia import ELECTRICITY_COLUMN, export, scenario_table
from l4m_lca.project import (DEFAULT_INDICATORS, category, ef31_methods, find_activity, method_units, open_project,
                             select_background)
from l4m_lca.regional import electricity_markets, regional_table

## Parameters

In [2]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BG = select_background("ecoinvent")
BACKGROUND = BG.name

INDICATORS = DEFAULT_INDICATORS            # EF 3.1 indicators (substrings of the method names)

# Background activities of each background database: (name, location)
ELECTRICITY = {"ecoinvent": ("market group for electricity, medium voltage", "RER"),
               "bafu": ("Electricity, medium voltage, at grid", "RER")}
# Shielding and protective gases: gas name -> activity. A use case selects its gas with "gas" (default "argon").
GASES = {"ecoinvent": {"argon": ("market for argon, liquid", "RER"),
                       "nitrogen": ("market for nitrogen, liquid", "RER")},
         "bafu": {"argon": ("Argon, liquid, at plant", "RER"),
                  "nitrogen": ("Nitrogen, liquid, at plant", "RER")}}
# Filler wire (aluminium ER4043): proxy wrought aluminium alloy, as in KPI_calculations.xlsx
FILLER = {"ecoinvent": ("market for aluminium, wrought alloy", "GLO"),
          "bafu": ("Aluminium, production mix, wrought alloy, at plant", "RER")}

# Prices of the main resource inputs
# Gas and filler prices as in KPI_calculations.xlsx (Air Products UK cylinder prices; weldmonger filler wire)
COST_FACTORS = {
    "electricity_EUR_per_kWh": EUROSTAT_ELEC_PRICE_2025S1["RER"],   # 0.164, Eurostat 2025-S1, band ID, EU27 average
    "argon_EUR_per_kg": 5.87,          # 10.47 EUR/m3 / 1.784 kg/m3
    "nitrogen_EUR_per_kg": 5.35,       # 6.69 EUR/m3 / 1.2506 kg/m3 (cylinder price; bulk nitrogen is much cheaper)
    "filler_EUR_per_kg": 26.17,        # aluminium filler wire ER4043
}

# Regional analysis: EU27 member states and the United Kingdom (GB).
# Industrial electricity price of each country (EUR/kWh); gases and filler keep their prices above.
# GB: DESNZ table 3.4.2, Medium band, 2025 Q1-Q2 (see l4m_lca/constants.py for the sources).
# Prices: REGIONAL_ELEC_PRICE, labour costs: regional_labour_costs() (file LABOUR_COST_FILE);
# both in l4m_lca. The EU27 average (REGIONAL_BASELINE = "RER") is the baseline.
REGIONAL_COUNTRIES = EU27_GB

# Labour cost per hour of each country (EUR/h): Eurostat lc_lci_lev 2025, NACE C (manufacturing),
# sheet "labour_cost" of country_data.xlsx. The EU27 average is the REGIONAL_BASELINE value.
# GB: Eurostat 2016 UK value x ONS ALCH manufacturing growth 2016-2025 (see l4m_lca/constants.py).
# Labour hours per joined unit = joining time x operators (parameter "operators", default 1 per station).
# Personnel costs = labour hours x labour cost. Nominal OPEX = consumable costs + personnel costs.
LABOUR_COSTS = regional_labour_costs()
LABOUR_COST = LABOUR_COSTS[REGIONAL_BASELINE]   # 35.0 EUR/h, EU27 average

# Power return rate of the lasers (laser output power / electrical power), KPI_calculations.xlsx (WMG data).
# power_kW_joining is laser output power; the model divides it by laser_efficiency. Not used for TIG or brazing.
LASER_EFFICIENCY = 0.34

In [3]:
# Joining parameters of the SoA processes:
# joining power and welding station power (kW), joining speed (m/s), joint length (m), gas flow (kg/s).
# minimum = lowest power and gas flow at the fastest speed; maximum = highest at the slowest speed.

SOA_JOINING_PARAMS = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.4, "max": 1.5},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.012, "max": 0.014},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.8, "max": 2.52},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.0025, "max": 0.0025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000268, "max": 0.000268},
        "filler_kg": {"min": 0.0765, "max": 0.0765},   # aluminium filler wire per unit (KPI_calculations.xlsx)
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.4, "max": 2.1},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.015, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000892, "max": 0.000892},
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.8, "max": 2.0},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.009, "max": 0.009},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.001487, "max": 0.001487},
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 0, "max": 0},
        "power_kW_station": {"min": 2000, "max": 2000},
        "joining_speed_m_per_s": {"min": 0.460, "max": 0.460},
        "joint_length_m": 27.5,
        "gas": "nitrogen",                                          # brazing furnace atmosphere: N2 250 m3/h
        "gas_flow_kg_per_s": {"min": 0.086847, "max": 0.086847},    # 250 m3/h * 1.2506 kg/m3 / 3600
    },
}

# SoA technology of each use case
SOA_TECHNOLOGY = {
    "Heat exchanger": "standard laser",
    "Fuel tank": "TIG",
    "Heterogeneous pipe": "standard laser",
    "Freeze Dryers Shelf": "standard laser",
    "Battery cooling plate": "controlled atmosphere brazing",
}

In [4]:
# Joining parameters with the L4M-solution. power_kW_station includes the chiller.
L4M_SOLUTION_JOINING_PARAMS = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.26, "max": 1.35},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.62, "max": 2.268},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
        "filler_kg": {"min": 0.0765, "max": 0.0765},   # aluminium filler wire per unit (KPI_calculations.xlsx)
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.26, "max": 1.89},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.62, "max": 1.8},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.1, "max": 0.1},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 8, "max": 10},
        "laser_efficiency": LASER_EFFICIENCY,
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joint_length_m": 27.5,
        "joining_speed_m_per_s": {"min": 0.4, "max": 0.4},
        "gas_flow_kg_per_s": {"min": 0, "max": 0},
    },
}

## Project and background data

In [5]:
background = open_project(BG.project, BG.database)
elec = find_activity(BG.database, *ELECTRICITY[BACKGROUND])
gases = {name: find_activity(BG.database, *spec) for name, spec in GASES[BACKGROUND].items()}
filler = find_activity(BG.database, *FILLER[BACKGROUND])
print(f"Background: {BG.database} (project {BG.project}), results in {BG.results}/")

methods = ef31_methods(INDICATORS)
units = method_units(methods)
for m in methods:
    print(f"  {m}  ->  unit: {units[category(m)]}")

impacts = BackgroundImpacts(methods, elec)
prices = {elec.key: COST_FACTORS["electricity_EUR_per_kWh"],
          gases["argon"].key: COST_FACTORS["argon_EUR_per_kg"],
          gases["nitrogen"].key: COST_FACTORS["nitrogen_EUR_per_kg"],
          filler.key: COST_FACTORS["filler_EUR_per_kg"]}

Background: ecoinvent-3.12-cutoff (project Lasers4MaaS), results in results/
  ('ecoinvent-3.12', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)')  ->  unit: mol H+-Eq
  ('ecoinvent-3.12', 'EF v3.1', 'eutrophication: terrestrial', 'accumulated exceedance (AE)')  ->  unit: mol N-Eq
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, inorganics', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, organics', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'particulate matter formation', 'impact on human health')  ->  unit: disease incidence
  ('ecoinvent-3.12', 'EF v3.1', 'photochemical oxidant formation: human health', 'tropospheric ozone concentration increase')  ->  unit: kg NMVOC-Eq
  ('ecoinvent-3.12', 'EN15804+A2 - Core impact ca

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


## Unit processes

Electricity and argon exchanges come from the parameters above. Each exchange has an amount (the midpoint) and a minimum and maximum for the scenarios.

In [6]:
soa = bd.Database(SOA_DATABASE)
soa.write(joining_processes(SOA_DATABASE, SOA_JOINING_PARAMS, SOA_TECHNOLOGY, elec, gases, filler=filler))

l4m = bd.Database(L4M_SOLUTION_DATABASE)
l4m.write(joining_processes(L4M_SOLUTION_DATABASE, L4M_SOLUTION_JOINING_PARAMS, L4M_SOLUTION_TECHNOLOGY, elec, gases, filler=filler))

100%|██████████| 5/5 [00:00<00:00, 14037.16it/s]

10:50:14+0100 [info     ] Vacuuming database            



100%|██████████| 5/5 [00:00<00:00, 32870.72it/s]


10:50:25+0100 [info     ] Vacuuming database            


## LCIA results

Impacts, foreground electricity and operating costs of each process in the minimum, amount and maximum scenarios.
The operating costs are the consumable costs, the personnel costs (EU27 average labour cost) and the nominal OPEX (their sum).

In [7]:
lcia_df = scenario_table([soa, l4m], impacts, prices, wage=LABOUR_COST)
export(lcia_df, units, BG.results, JOINING_STEM)
lcia_df

Results saved to results/State_of_the_Art_L4M_solution_LCIA_results.csv and results/State_of_the_Art_L4M_solution_LCIA_results.json


,Database,Process,Key,Scenario,acidification,eutrophication: terrestrial,human toxicity: carcinogenic,"human toxicity: carcinogenic, inorganics","human toxicity: carcinogenic, organics",particulate matter formation,photochemical oxidant formation: human health,climate change: total (EF v3.1 - IPCC 2021),Electricity Consumption (kWh),Consumable costs (EUR),Personnel costs (EUR),Nominal OPEX (EUR)
0,State_of_the_Art.bbb12d81,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,0.061096,0.093680,2.085724e-09,1.316949e-09,7.687746e-10,2.128953e-07,0.029315,13.039440,33.212560,33.223740,0.581220,33.804960
1,State_of_the_Art.bbb12d81,Battery cooling plate joined with controlled a...,Battery cooling plate,amount,0.061096,0.093680,2.085724e-09,1.316949e-09,7.687746e-10,2.128953e-07,0.029315,13.039440,33.212560,33.223740,0.581220,33.804960
2,State_of_the_Art.bbb12d81,Battery cooling plate joined with controlled a...,Battery cooling plate,maximum,0.061096,0.093680,2.085724e-09,1.316949e-09,7.687746e-10,2.128953e-07,0.029315,13.039440,33.212560,33.223740,0.581220,33.804960
3,State_of_the_Art.bbb12d81,Freeze Dryers Shelf joined with standard laser,Freeze Dryers Shelf,minimum,0.031697,0.048607,1.091721e-09,6.850246e-10,4.066965e-10,1.102408e-07,0.015267,6.845258,10.997603,14.605686,14.259259,28.864945
4,State_of_the_Art.bbb12d81,Freeze Dryers Shelf joined with standard laser,Freeze Dryers Shelf,amount,0.031899,0.048916,1.098613e-09,6.893828e-10,4.092302e-10,1.109387e-07,0.015364,6.888413,11.117429,14.625337,14.259259,28.884596
5,State_of_the_Art.bbb12d81,Freeze Dryers Shelf joined with standard laser,Freeze Dryers Shelf,maximum,0.032101,0.049225,1.105505e-09,6.937410e-10,4.117639e-10,1.116365e-07,0.015461,6.931568,11.237255,14.644988,14.259259,28.904248
6,State_of_the_Art.bbb12d81,Fuel tank joined with TIG,Fuel tank,minimum,0.032040,0.050004,1.499756e-09,6.778931e-10,8.218625e-10,1.737780e-07,0.015451,6.458750,13.212222,7.352885,19.677778,27.030663
7,State_of_the_Art.bbb12d81,Fuel tank joined with TIG,Fuel tank,amount,0.032381,0.050525,1.511397e-09,6.852546e-10,8.261422e-10,1.749568e-07,0.015614,6.531644,13.414622,7.386079,19.677778,27.063857
8,State_of_the_Art.bbb12d81,Fuel tank joined with TIG,Fuel tank,maximum,0.032723,0.051047,1.523038e-09,6.926161e-10,8.304219e-10,1.761356e-07,0.015777,6.604538,13.617022,7.419272,19.677778,27.097050
9,State_of_the_Art.bbb12d81,Heat exchanger joined with standard laser,Heat exchanger,minimum,0.086497,0.132837,2.990427e-09,1.870955e-09,1.119472e-09,3.016599e-07,0.041787,18.759850,21.975735,51.738188,29.791667,81.529855


In [8]:
all_units = {**units, ELECTRICITY_COLUMN: "kWh", **COST_UNITS}
indicator_cols = [c for c in lcia_df.columns if c not in ("Database", "Process", "Key", "Scenario")]
display(plots.scenario_bars(lcia_df, list(SOA_JOINING_PARAMS), [SOA_DATABASE, L4M_SOLUTION_DATABASE], all_units,
                            technology_of, indicator_cols))

### Operating costs per joined unit

Consumable costs, personnel costs and nominal OPEX (EUR) in each scenario.

In [9]:
lcia_df.assign(Technology=lcia_df["Process"].map(technology_of)).pivot_table(
    index=["Key", "Technology"], columns="Scenario", values=list(COST_COLUMNS), sort=False)[
    [(col, s) for col in COST_COLUMNS for s in ("minimum", "amount", "maximum")]].round(4)

Consumable costs (EUR)  \
Scenario                                                           minimum   
Key                   Technology                                             
Battery cooling plate controlled atmosphere brazing                33.2237   
                      L4M-solution                                  0.1417   
Freeze Dryers Shelf   standard laser                               14.6057   
                      L4M-solution                                  0.6202   
Fuel tank             TIG                                           7.3529   
                      L4M-solution                                  2.9529   
Heat exchanger        standard laser                               51.7382   
                      L4M-solution                                 28.9412   
Heterogeneous pipe    standard laser                                0.0241   
                      L4M-solution                                  0.0175   

                                                                       \
Scenario                                              amount  maximum   
Key                   Technology                                        
Battery cooling plate controlled atmosphere brazing  33.2237  33.2237   
                      L4M-solution                    0.1509   0.1601   
Freeze Dryers Shelf   standard laser                 14.6253  14.6450   
                      L4M-solution                    0.6218   0.6234   
Fuel tank             TIG                             7.3861   7.4193   
                      L4M-solution                    2.9617   2.9705   
Heat exchanger        standard laser                 56.0737  60.4091   
                      L4M-solution                   28.9515  28.9619   
Heterogeneous pipe    standard laser                  0.0324   0.0408   
                      L4M-solution                    0.0176   0.0178   

                                                    Personnel costs (EUR)  \
Scenario                                                          minimum   
Key                   Technology                                            
Battery cooling plate controlled atmosphere brazing                0.5812   
                      L4M-solution                                 0.6684   
Freeze Dryers Shelf   standard laser                              14.2593   
                      L4M-solution                                 1.2833   
Fuel tank             TIG                                         19.6778   
                      L4M-solution                                 1.9678   
Heat exchanger        standard laser                              29.7917   
                      L4M-solution                                16.6833   
Heterogeneous pipe    standard laser                               0.0366   
                      L4M-solution                                 0.0366   

                                                                       \
Scenario                                              amount  maximum   
Key                   Technology                                        
Battery cooling plate controlled atmosphere brazing   0.5812   0.5812   
                      L4M-solution                    0.6684   0.6684   
Freeze Dryers Shelf   standard laser                 14.2593  14.2593   
                      L4M-solution                    1.2833   1.2833   
Fuel tank             TIG                            19.6778  19.6778   
                      L4M-solution                    1.9678   1.9678   
Heat exchanger        standard laser                 32.2743  34.7569   
                      L4M-solution                   16.6833  16.6833   
Heterogeneous pipe    standard laser                  0.0487   0.0609   
                      L4M-solution                    0.0366   0.0366   

                                                    Nominal OPEX (EUR)  \
Scenario                                                     

## Regional analysis: EU and UK electricity markets

Replaces the RER electricity market with the market of each EU member state and of the United Kingdom (GB). Each country also gets its own labour cost (personnel costs).
The map shows the impacts and operating costs per joined unit.
Eurostat has no 2025 labour cost for some countries (for example NL: confidential, BE: not available). Their personnel costs and nominal OPEX are empty (NaN).
Eurostat has no recent UK data. The GB electricity price and labour cost come from UK sources (DESNZ, ONS), converted with ECB rates. The sources are in `l4m_lca/constants.py`.
The file `results/regional_joining.csv` is an input of `workflow_assembly.ipynb`.

In [10]:
markets = electricity_markets(background, REGIONAL_COUNTRIES, BG.country_electricity)
regional_df = regional_table([soa, l4m], impacts, markets, REGIONAL_ELEC_PRICE, prices, REGIONAL_BASELINE,
                           wages=LABOUR_COSTS)
regional_df["technology"] = regional_df["process"].map(technology_of)
regional_df.to_csv(f"{BG.results}/regional_joining.csv", index=False)
regional_df.head()

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


,database,process,key,scenario,country,indicator,impact,technology
0,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,acidification,6.109622e-02,controlled atmosphere brazing
1,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,eutrophication: terrestrial,9.368015e-02,controlled atmosphere brazing
2,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,human toxicity: carcinogenic,2.085724e-09,controlled atmosphere brazing
3,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,"human toxicity: carcinogenic, inorganics",1.316949e-09,controlled atmosphere brazing
4,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,"human toxicity: carcinogenic, organics",7.687746e-10,controlled atmosphere brazing


In [11]:
amount_df = regional_df[regional_df["scenario"] == "amount"]
display(plots.regional_explorer(
    amount_df,
    filters=[("Use case", "key", list(SOA_JOINING_PARAMS), None),
             ("Technology", "database", [SOA_DATABASE, L4M_SOLUTION_DATABASE], None)],
    units=all_units,
    title=lambda f: f"{f['key']}, {f['database'].replace('_', ' ')}: per joined unit",
    baseline=REGIONAL_BASELINE, countries=REGIONAL_COUNTRIES,
))